# B2-023 — Review and Self-Quiz

*60 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261015`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`, `B2-021-cross-modal-transformers-vision`, `B2-022-probabilistic-latent-models`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../B2-020-language-transformers/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../B2-021-cross-modal-transformers-vision/lesson.ipynb), [B2-022-probabilistic-latent-models](../B2-022-probabilistic-latent-models/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

## Formula and protocol sheet

- **Sigmoid and BCE:** $\sigma(\ell)=1/(1+e^{-\ell})$, $\sigma(-\ell)=1-\sigma(\ell)$, $\sigma'=\sigma(1-\sigma)$; $-\log\sigma(\ell)=\log(1+e^{-\ell})$; always `F.binary_cross_entropy_with_logits` on raw logits (mean reduction).
- **Value function:** $V(D,G)=E_{p_{\text{data}}}[\log D]+E_z[\log(1-D(G(z)))]$; $\mathcal L_D=\mathrm{BCE}(\text{real},1)+\mathrm{BCE}(\text{fake},0)$ (sum of two means); confused discriminator: $\mathcal L_D=2\log2$.
- **Optimal discriminator:** maximize $a\log y+b\log(1-y)$ pointwise: $D^*=p_{\text{data}}/(p_{\text{data}}+p_g)$ when both are positive; boundary convention $D^*\in\{0,1\}$ otherwise.
- **JS:** $M=\frac12(P+Q)$, $\mathrm{JS}=\frac12\mathrm{KL}(P\|M)+\frac12\mathrm{KL}(Q\|M)\in[0,\log2]$, zero iff $P=Q$; $C(G)=-\log4+2\,\mathrm{JS}(p_{\text{data}}\|p_g)$.
- **Non-saturating generator loss:** $\mathrm{BCE}(\text{fake},1)$; logit gradient $-(1-\sigma(\ell))$ versus the saturating $-\sigma(\ell)$.
- **Training a GAN:** `d_step(G, D, opt_d, real, z)` with `G(z).detach()`, then `g_step(G, D, opt_g, z)`; each starts with `zero_grad(set_to_none=True)`; two optimizers.
- **Mode coverage:** nearest centre (lowest index on ties), counted iff squared distance $\le r^2$, covered iff count $\ge\lceil n/(8K)\rceil$.
- **Schedule:** $(T+1)$-length float64 tensors with padding $\beta_0=0$, $\alpha_0=\bar\alpha_0=1$; $\alpha_t=1-\beta_t$, $\bar\alpha_t=\prod_{s\le t}\alpha_s$, $\tilde\beta_t=\frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t}\beta_t$ ($\tilde\beta_1=0$).
- **Forward process:** $x_t=\sqrt{\alpha_t}x_{t-1}+\sqrt{\beta_t}\varepsilon_t$; $x_t=\sqrt{\bar\alpha_t}x_0+\sqrt{1-\bar\alpha_t}\varepsilon$; sum rule $N(0,s^2)+N(0,r^2)=N(0,s^2+r^2)$ for independent terms.
- **Equal-variance KL:** $\lVert\mu_1-\mu_2\rVert^2/(2\sigma^2)$; with $\sigma_t^2=\tilde\beta_t$ each $t\ge2$ term is $\lambda_t\lVert\varepsilon-\varepsilon_\theta\rVert^2$; $t=1$ is a surrogate.
- **Training a DDPM:** draw `t` in $1..T$ then `eps`; `x_t = q_sample(x0, t, eps, alpha_bar)`; loss `((model(x_t, t/T) - eps) ** 2).mean()`.
- **Sampling:** mean $=\frac{1}{\sqrt{\alpha_t}}\big(x_t-\frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\hat\varepsilon\big)$; add $\sqrt{\tilde\beta_t}z$ for $t\ge2$, nothing at $t=1$; loop $t=T,\dots,1$ with `noises[t - 1]`.
- **Latent diffusion:** $z_0=s\cdot\text{encode}(x)$, $\hat x=\text{decode}(\hat z_0/s)$, $s=0.25$ here ($0.18215$ in Stable Diffusion v1).
- **Cross-attention:** $Q$ from the latent token, $K,V$ from the condition tokens; drop a token by a $-\infty$ score.
- **Guidance:** $\hat\varepsilon=\varepsilon_{\text{uncond}}+w(\varepsilon_{\text{cond}}-\varepsilon_{\text{uncond}})$, with conditioning dropout $0.2$ in training.

## Concept retrieval table

| Concept | Recall prompt | Redo |
|---|---|---|
| generative-adversarial-network | Derive $D^*$; prove the JS identity; implement both losses and an alternating step; certify coverage; diagnose failures. | p01, p02, p06, p07, p08, p13, p14, p17, p21, p22 |
| denoising-diffusion-probabilistic-models | Compute $\bar\alpha_t$; prove the closed form; derive the ε objective; implement and train; sample; repair a broken sampler. | p03, p04, p09, p10, p11, p15, p16, p18, p19, p21, p23 |
| stable-diffusion | Map the pipeline; implement guidance and cross-attention; train the latent analogue; audit a pipeline. | p05, p12, p20, p21, p24 |

## Self-quiz

1. Compute $\sigma(\ln4)$ and the BCE of logit $\ln4$ with label $0$.
2. Give $D^*(x)$ when $p_{\text{data}}(x)=\frac14$ and $p_g(x)=\frac1{12}$, in normal form.
3. Why is the discriminator loss $2\log2$ at the confused discriminator?
4. State $\mathrm{JS}(P\|Q)$ for $P=(1,0)$, $Q=(\frac12,\frac12)$ to six decimals.
5. Why does the saturating generator loss stall early in training?
6. Which probe detects a missing `detach` in the D-step?
7. Name one trace signature of mode collapse and one of discriminator overpowering.
8. For $\beta=(0.1,0.5,0.2)$, give $\bar\alpha_1,\bar\alpha_2,\bar\alpha_3$.
9. Why does `q_sample` with `alpha` instead of `alpha_bar` pass a probe at $t=1$?
10. Give $\tilde\beta_1$ and explain what it means for the last sampling step.
11. What are the ε-network's input and regression target?
12. Why does the equal-variance KL argument exclude $t=1$?
13. Why does latent diffusion need a scaling factor, and where is it applied?
14. In cross-attention conditioning, which side supplies the queries, and how is a class token removed?
15. Compute the guided prediction for $\varepsilon_{\text{uncond}}=(0,1)$, $\varepsilon_{\text{cond}}=(1,1)$, $w=1.5$, and say what $w=0$ and $w=1$ return.

## Common pitfalls, Exam connections, and forward boundary

Common pitfalls: sigmoid-then-log, a sigmoid inside the discriminator, swapped BCE labels, the saturating generator loss, a missing `detach`, one optimizer for two players, reading losses as quality, `<` at the coverage radius, $\alpha$ for $\bar\alpha$, variance used as standard deviation, off-by-one schedules and loops, noise at the final step, an $x_0$ regression target, a missing latent scaling factor, zero-score token "removal", sign-flipped guidance, and held-out rows in a training or conditioning set.
Exam connections: Round 2 style tasks ask for an exact normal-form value, a guarded implementation matched to literal probes, a proof with its conditions, a training certificate with stated tolerances, or the first broken invariant in a pipeline.
Forward-only topics (Wasserstein GANs, DCGAN, score-based diffusion, fast samplers, and real Stable Diffusion on a GPU in `B2-024`) are not required or permitted shortcuts here.

## Collected answers

1. $\sigma(\ln4)=\frac{4}{5}$; BCE $=-\log(1-\frac45)=\log5=1.609438$.
2. $\frac{1/4}{1/4+1/12}=\frac{3/12}{4/12}=\frac34$ ($a=3$, $b=4$).
3. Each of the two BCE means is $-\log\frac12=\log2$ when every logit is $0$, and the loss is their sum.
4. $M=(\frac34,\frac14)$; $\mathrm{KL}(P\|M)=\log\frac43=0.287682$; $\mathrm{KL}(Q\|M)=\frac12\log\frac23+\frac12\log2=0.143841$; $\mathrm{JS}=0.215762$.
5. Its logit gradient is $-\sigma(\ell)$, which is nearly $0$ when the discriminator confidently rejects fakes ($\ell\ll0$).
6. Every generator parameter's `.grad` is `None` immediately after `d_step`.
7. Mode collapse: balanced-looking losses with coverage counts concentrated on one or two components. Overpowering: $\mathcal L_D$ near $0$ and a large, growing non-saturating $\mathcal L_G$.
8. $0.9$, $0.45$, $0.36$.
9. Because $\bar\alpha_1=\alpha_1$; the defect only shows at $t\ge2$.
10. $\tilde\beta_1=0$: the posterior is a point mass at $x_0$, so the final step returns the mean with no noise.
11. Input $(x_t,t/T)$; target the noise $\varepsilon$ that produced $x_t$.
12. $\tilde\beta_1=0$, so the KL would divide by a zero variance; the bound's $t=1$ term is a separate decoder likelihood, and the ε loss there is a surrogate.
13. The schedule assumes data of unit scale or smaller; multiply by $s$ after encoding (training) and divide by $s$ before decoding (generation).
14. The latent (with $t/T$) supplies the queries; the class token is removed by setting its attention score to $-\infty$ so the null token gets weight $1$.
15. $(0,1)+1.5(1,0)=(1.5,1)$; $w=0$ returns $\varepsilon_{\text{uncond}}=(0,1)$ and $w=1$ returns $\varepsilon_{\text{cond}}=(1,1)$.